This notebook contains descriptive data analysis and pre-processing and application of ML techniques to predict hospital infection by fungus or bacteria among COVID-19 positeve patients.


In [17]:
# Última modificação: 20/02/2025

In [1]:
!pip install tensorflow
!pip install shap
!pip install seaborn
!pip install imblearn
!pip install 'openpyxl>=3.0.0'

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
# My utils
from mypipeline import *


# Libraries
import numpy as np
import pandas as pd
import random as rd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime as dt
import os
import time
import math

from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression, SelectKBest, SelectPercentile
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, make_scorer, roc_auc_score, recall_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.under_sampling import RandomUnderSampler


# MICE, KNN, Dumb
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer


2025-03-29 14:25:37.812616: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 14:25:42.564283: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-29 14:25:42.564316: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-29 14:25:42.638035: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-29 14:25:53.861070: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 14:25:53.910499: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

# 1.0 Read original dataset

In [3]:
ls

Autoencoders/          HSL.csv  ML_notebook.ipynb  __pycache__/
COVID-PROGNOSTIC.csv*  LICENSE  mypipeline.py      README.md
HBP.csv                LPMD.py  output_excel/      results/
HCAI-INFECTION.xlsx*   main.py  plots/             results_example/


In [4]:
original_dataset = pd.read_excel("HCAI-INFECTION.xlsx")

original_dataset = original_dataset.drop(columns='Unnamed: 0')

original_dataset

,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,INFEC,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo
3,01A30B6624DDB49F16CA4311CC37D65F,38,25.0,10.0,0.10,0.09,33.8,60.0,NaN,0.98,...,13.0,142.0,NaN,NaN,25.0,83.0,9.4,1,0,negativo
4,02367C393B8487123744A46CFBB91A28,74,13.0,10.0,0.18,0.12,33.1,72.0,1.14,0.80,...,13.1,135.0,1.10,1.20,39.0,94.5,9.7,1,0,negativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,FE69345DBB6AE27A1E81775EE898A25B,65,23.0,10.0,0.25,0.27,34.1,225.0,1.26,0.77,...,13.0,138.0,1.30,1.03,26.0,89.7,11.1,1,0,negativo
495,FECC5CE1CFE3BCE881F29C2333527135,42,10.0,40.0,NaN,NaN,33.1,NaN,NaN,0.86,...,12.9,139.0,NaN,NaN,29.0,92.5,10.4,0,0,negativo
496,FF19A1D8C1EB3A7A73541F3443B4FA00,79,20.0,40.0,0.21,0.22,32.5,170.0,1.22,0.70,...,15.2,142.0,1.20,1.05,27.0,99.2,11.4,0,0,negativo
497,FF4B2EED093AE641B9328FDB293C4116,53,40.0,0.0,0.16,0.25,33.3,NaN,1.20,1.05,...,12.7,140.0,0.97,1.06,25.0,84.1,11.0,1,0,negativo


In [5]:
original_dataset.columns

Index(['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'Basófilos', 'Bilirrubina Direta',
       'Bilirrubina Indireta', 'CHCM', 'CK', 'Calcio Ionizavel', 'Creatinina',
       'DHL', 'Dimeros D, quant', 'Eosinófilos', 'Eritrócitos, urina',
       'Fibrinogenio', 'Fosfatase Alcalina', 'Gama-GT', 'Glicose', 'HCM',
       'HCO3 venoso', 'Hemoglobina', 'Leucócitos', 'Leucócitos, urina',
       'Linfócitos', 'Magnésio', 'Monócitos', 'Neutrófilos', 'Plaquetas',
       'Potássio', 'Proteína C-Reativa', 'RDW', 'Sódio', 'TP_INR',
       'TTPA - Paciente_Normal', 'Uréia', 'VCM', 'Volume plaquetário médio',
       'SEXO', 'INFEC', 'Infecção Hospitalar'],
      dtype='object')

In [6]:
original_dataset.columns = ['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'Basófilos', 'Bilirrubina Direta',
       'Bilirrubina Indireta', 'CHCM', 'CK', 'Calcio Ionizavel', 'Creatinina',
       'DHL', 'Dimeros D, quant', 'Eosinófilos', 'Eritrócitos, urina',
       'Fibrinogenio', 'Fosfatase Alcalina', 'Gama-GT', 'Glicose', 'HCM',
       'HCO3 venoso', 'Hemoglobina', 'Leucócitos', 'Leucócitos, urina',
       'Linfócitos', 'Magnésio', 'Monócitos', 'Neutrófilos', 'Plaquetas',
       'Potássio', 'Proteína C-Reativa', 'RDW', 'Sódio', 'TP_INR',
       'TTPA - Paciente_Normal', 'Uréia', 'VCM', 'Volume plaquetário médio',
       'SEXO', 'TARGET', 'Infecção Hospitalar']

original_dataset

,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo
3,01A30B6624DDB49F16CA4311CC37D65F,38,25.0,10.0,0.10,0.09,33.8,60.0,NaN,0.98,...,13.0,142.0,NaN,NaN,25.0,83.0,9.4,1,0,negativo
4,02367C393B8487123744A46CFBB91A28,74,13.0,10.0,0.18,0.12,33.1,72.0,1.14,0.80,...,13.1,135.0,1.10,1.20,39.0,94.5,9.7,1,0,negativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,FE69345DBB6AE27A1E81775EE898A25B,65,23.0,10.0,0.25,0.27,34.1,225.0,1.26,0.77,...,13.0,138.0,1.30,1.03,26.0,89.7,11.1,1,0,negativo
495,FECC5CE1CFE3BCE881F29C2333527135,42,10.0,40.0,NaN,NaN,33.1,NaN,NaN,0.86,...,12.9,139.0,NaN,NaN,29.0,92.5,10.4,0,0,negativo
496,FF19A1D8C1EB3A7A73541F3443B4FA00,79,20.0,40.0,0.21,0.22,32.5,170.0,1.22,0.70,...,15.2,142.0,1.20,1.05,27.0,99.2,11.4,0,0,negativo
497,FF4B2EED093AE641B9328FDB293C4116,53,40.0,0.0,0.16,0.25,33.3,NaN,1.20,1.05,...,12.7,140.0,0.97,1.06,25.0,84.1,11.0,1,0,negativo


In [7]:
dataset1 = original_dataset.copy(deep=True)

print(dataset1.shape)

dataset1.head(3)

(499, 40)


,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo


In [8]:
dataset1["TARGET"].value_counts()

TARGET
0    466
1     33
Name: count, dtype: int64

In [9]:
dataset2 = pd.read_csv("COVID-PROGNOSTIC.csv")

dataset2 = dataset2.drop(columns="Unnamed: 0")

dataset2.columns

Index(['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'AST (TGO)', 'Basófilos',
       'Basófilos (%)', 'CHCM', 'Creatinina', 'Eosinófilos', 'Eosinófilos (%)',
       'Eritrócitos', 'HCM', 'Hematócrito', 'Hemoglobina', 'Leucócitos',
       'Linfócitos', 'Linfócitos (%)', 'Monócitos', 'Monócitos (%)',
       'Neutrófilos', 'Neutrófilos (%)', 'Plaquetas', 'Potássio',
       'Proteína C-Reativa', 'RDW', 'Sódio', 'Uréia', 'VCM',
       'Volume plaquetário médio', 'SEXO', 'GRAVIDADE'],
      dtype='object')

In [10]:
dataset2.columns = ['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'AST (TGO)', 'Basófilos',
       'Basófilos (%)', 'CHCM', 'Creatinina', 'Eosinófilos', 'Eosinófilos (%)',
       'Eritrócitos', 'HCM', 'Hematócrito', 'Hemoglobina', 'Leucócitos',
       'Linfócitos', 'Linfócitos (%)', 'Monócitos', 'Monócitos (%)',
       'Neutrófilos', 'Neutrófilos (%)', 'Plaquetas', 'Potássio',
       'Proteína C-Reativa', 'RDW', 'Sódio', 'Uréia', 'VCM',
       'Volume plaquetário médio', 'SEXO', 'TARGET']

dataset2

,ID_PACIENTE,Idade,ALT (TGP),AST (TGO),Basófilos,Basófilos (%),CHCM,Creatinina,Eosinófilos,Eosinófilos (%),...,Plaquetas,Potássio,Proteína C-Reativa,RDW,Sódio,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET
0,00017961865C4F766FDBB3CD8FE0BFB0,54,26.0,24.0,40.0,0.6,34.5,1.02,60.0,0.9,...,176000.0,4.0,0.11,13.1,138.0,35.0,86.0,9.8,1,0
1,000F0BC139D2846DB86AA32B8F05B215,41,NaN,NaN,20.0,0.4,33.2,1.04,160.0,3.4,...,279000.0,4.3,NaN,14.0,142.0,33.0,83.2,10.0,1,0
2,0028785949D91BD93442838FC898E229,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,002B919CC409B11DE52FB212379BE2CB,41,26.0,26.0,40.0,0.6,32.9,0.73,240.0,3.4,...,275000.0,NaN,0.12,13.3,NaN,30.0,88.3,11.2,0,0
4,003051C9B19101D1C10C5DC654384017,36,27.0,18.0,10.0,0.3,33.4,0.97,30.0,0.9,...,231000.0,4.2,0.08,14.4,139.0,16.0,83.7,10.4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4315,FFABB233208E1E66DDC025BC5CC2E4D2,68,18.0,19.0,70.0,1.1,34.7,1.19,710.0,11.5,...,260000.0,NaN,0.13,12.1,NaN,53.0,85.3,9.9,1,0
4316,FFB440876DC059A0359FEFBA1D6FB28C,20,22.0,30.0,30.0,0.5,31.4,0.95,130.0,2.2,...,286000.0,3.6,0.03,12.7,138.0,28.0,85.3,11.3,1,0
4317,FFEC3898BAA04751EB00C108270B8F7E,39,35.0,23.0,40.0,0.4,34.5,0.91,190.0,2.1,...,147000.0,4.6,0.63,12.4,139.0,32.0,85.3,9.9,1,0
4318,FFF5753408C98D5E0218931420B6AF85,17,52.0,47.0,20.0,0.2,33.8,0.78,200.0,2.1,...,247000.0,4.0,0.07,13.2,139.0,22.0,87.3,11.2,0,0


In [11]:
dataset2["TARGET"].value_counts()

TARGET
0    3926
1     394
Name: count, dtype: int64

In [12]:
ls

Autoencoders/          HSL.csv  ML_notebook.ipynb  __pycache__/
COVID-PROGNOSTIC.csv*  LICENSE  mypipeline.py      README.md
HBP.csv                LPMD.py  output_excel/      results/
HCAI-INFECTION.xlsx*   main.py  plots/             results_example/


In [13]:
# Dataset 3

dataset3 = pd.read_csv("HSL.csv")

dataset3.columns = ['sex', 'age', 'creatinine', 'creatine phosphokinase', 'd-dimer',
       'eosinophils (%)', 'hemoglobin', 'leukocytes', 'lymphocytes (%)',
       'monocytes (%)', 'neutrophils (%)', 'platelets', 'potassium',
       'c-reactive protein', 'sodium', 'AST', 'ALT', 'troponin', 'urea',
       'TARGET']

dataset3["TARGET"].value_counts()

dataset3

,sex,age,creatinine,creatine phosphokinase,d-dimer,eosinophils (%),hemoglobin,leukocytes,lymphocytes (%),monocytes (%),neutrophils (%),platelets,potassium,c-reactive protein,sodium,AST,ALT,troponin,urea,TARGET
0,0,64.0,0.81,NaN,1297.0,0.0,13.2,5860.0,17.2,6.3,76.5,416000.0,4.4,8.77,136.0,NaN,NaN,0.16,23.0,0
1,0,NaN,1.49,NaN,NaN,2.5,12.3,5890.0,13.1,17.8,66.4,156000.0,3.7,0.96,137.0,17.0,24.0,NaN,57.0,0
2,0,52.0,1.04,124.0,233.0,0.0,15.1,4840.0,11.2,4.8,84.0,123000.0,4.0,1.49,137.0,88.0,182.0,0.16,37.0,0
3,0,76.0,0.95,73.0,643.0,0.0,12.9,8160.0,14.6,4.4,80.5,493000.0,3.9,0.52,140.0,30.0,24.0,0.16,45.0,0
4,0,88.0,2.09,132.0,337.0,0.0,12.0,10970.0,0.7,3.5,95.7,264000.0,4.5,5.45,135.0,30.0,20.0,0.16,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,0,58.0,0.54,NaN,NaN,0.0,13.3,6400.0,2.5,1.7,95.8,66000.0,4.3,0.17,140.0,NaN,NaN,NaN,57.0,1
1428,0,44.0,1.22,NaN,NaN,2.6,14.4,10690.0,30.0,7.3,59.7,276000.0,3.6,0.36,141.0,24.0,38.0,NaN,40.0,0
1429,0,51.0,0.96,NaN,226.0,2.3,14.2,3040.0,26.0,16.4,55.0,118000.0,4.2,0.97,140.0,27.0,45.0,NaN,25.0,0
1430,0,67.0,1.06,NaN,1005.0,7.2,13.6,6000.0,40.3,7.8,44.4,255000.0,3.7,4.54,139.0,27.0,24.0,NaN,43.0,0


In [14]:
dataset3.columns

Index(['sex', 'age', 'creatinine', 'creatine phosphokinase', 'd-dimer',
       'eosinophils (%)', 'hemoglobin', 'leukocytes', 'lymphocytes (%)',
       'monocytes (%)', 'neutrophils (%)', 'platelets', 'potassium',
       'c-reactive protein', 'sodium', 'AST', 'ALT', 'troponin', 'urea',
       'TARGET'],
      dtype='object')

In [15]:
dataset3["TARGET"].value_counts()

TARGET
0    903
1    529
Name: count, dtype: int64

In [16]:
dataset4 = pd.read_csv("HBP.csv")

dataset4.columns = ['sex', 'age', 'creatinine', 'creatine phosphokinase', 'd-dimer',
       'eosinophils (%)', 'hemoglobin', 'leukocytes', 'lymphocytes (%)',
       'monocytes (%)', 'neutrophils (%)', 'platelets', 'potassium',
       'c-reactive protein', 'sodium', 'AST', 'ALT', 'troponin', 'urea',
       'TARGET']

print(dataset4["TARGET"].value_counts())

dataset4

TARGET
0    118
1     67
Name: count, dtype: int64


,sex,age,creatinine,creatine phosphokinase,d-dimer,eosinophils (%),hemoglobin,leukocytes,lymphocytes (%),monocytes (%),neutrophils (%),platelets,potassium,c-reactive protein,sodium,AST,ALT,troponin,urea,TARGET
0,1,78.0,0.57,120.0,999.0,0.0,11.5,9650.0,7.9,4.1,88.0,301000.0,3.6,21.48,138.0,36.0,39.0,NaN,18.0,0
1,0,37.0,0.69,146.0,892.0,0.0,14.3,6430.0,11.5,2.8,85.7,135000.0,4.0,30.08,136.0,59.0,53.0,NaN,20.0,1
2,0,35.0,0.98,111.0,NaN,0.0,13.8,1780.0,32.0,9.6,58.4,15000.0,3.7,3.01,139.0,NaN,NaN,NaN,28.0,1
3,0,63.0,1.37,1238.0,2004.0,0.0,12.3,20160.0,2.4,3.4,NaN,262000.0,4.5,12.32,142.0,41.0,52.0,1.11,60.0,1
4,0,38.0,0.74,284.0,1757.0,0.4,13.4,7660.0,18.7,7.2,73.3,276000.0,3.8,7.78,138.0,75.0,95.0,0.16,18.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,1,86.0,0.90,26.0,1689.0,0.3,9.8,3930.0,16.8,7.9,74.7,176000.0,4.3,4.22,135.0,49.0,61.0,0.16,58.0,1
181,0,64.0,4.77,790.0,3559.0,0.0,11.6,NaN,2.1,4.1,NaN,231000.0,NaN,16.08,140.0,66.0,37.0,0.16,NaN,1
182,0,60.0,1.41,51.0,2118.0,0.2,15.5,10610.0,7.9,5.5,85.7,259000.0,5.0,30.45,137.0,74.0,94.0,0.16,59.0,1
183,0,53.0,1.05,43.0,585.0,0.2,15.6,6130.0,10.1,9.0,80.5,190000.0,3.6,17.39,138.0,41.0,49.0,0.16,23.0,1


In [ ]:
cd 

/home/filipe/Documentos/GitHub/Doutorado-real-datasets/results


In [21]:
ls

hosp1_KNN_auc.csv     hosp2_LPMD_spe.csv    hosp3_PMIVAE_sen.csv
hosp1_KNN_sen.csv     hosp2_Mean_auc.csv    hosp3_PMIVAE_spe.csv
hosp1_KNN_spe.csv     hosp2_Mean_sen.csv    hosp3_SAEI_auc.csv
hosp1_LPMD2_auc.csv   hosp2_Mean_spe.csv    hosp3_SAEI_sen.csv
hosp1_LPMD2_sen.csv   hosp2_MICE_auc.csv    hosp3_SAEI_spe.csv
hosp1_LPMD2_spe.csv   hosp2_MICE_sen.csv    hosp4_KNN_auc.csv
hosp1_LPMD_auc.csv    hosp2_MICE_spe.csv    hosp4_KNN_sen.csv
hosp1_LPMD_sen.csv    hosp2_PMIVAE_auc.csv  hosp4_KNN_spe.csv
hosp1_LPMD_spe.csv    hosp2_PMIVAE_sen.csv  hosp4_LPMD2_auc.csv
hosp1_Mean_auc.csv    hosp2_PMIVAE_spe.csv  hosp4_LPMD2_sen.csv
hosp1_Mean_sen.csv    hosp2_SAEI_auc.csv    hosp4_LPMD2_spe.csv
hosp1_Mean_spe.csv    hosp2_SAEI_sen.csv    hosp4_LPMD_auc.csv
hosp1_MICE_auc.csv    hosp2_SAEI_spe.csv    hosp4_LPMD_sen.csv
hosp1_MICE_sen.csv    hosp3_KNN_auc.csv     hosp4_LPMD_spe.csv
hosp1_MICE_spe.csv    hosp3_KNN_sen.csv     hosp4_Mean_auc.csv
hosp1_PMIVAE_auc.csv  hosp3_KNN_spe.csv     hosp4_M

In [25]:
import os
import pandas as pd

def consolidar_planilhas(pasta_arquivos, arquivo_saida):
    arquivos = [f for f in os.listdir(pasta_arquivos) if f.endswith('.csv')]
    
    dados_consolidados = []
    
    for arquivo in arquivos:
        caminho_arquivo = os.path.join(pasta_arquivos, arquivo)
        df = pd.read_csv(caminho_arquivo)
        df['Arquivo'] = arquivo  # Adiciona uma coluna com o nome do arquivo
        dados_consolidados.append(df)
    
    df_final = pd.concat(dados_consolidados, ignore_index=True)
    df_final.to_csv(arquivo_saida, index=False)
    print(f"Arquivo consolidado salvo em: {arquivo_saida}")

# Exemplo de uso
pasta = "/home/filipe/Documentos/GitHub/Doutorado-real-datasets/results"
saida = "saida_consolidada.csv"
consolidar_planilhas(pasta, saida)

Arquivo consolidado salvo em: saida_consolidada.csv


In [30]:
import os
import pandas as pd

def extrair_info_arquivo(nome_arquivo):
    partes = nome_arquivo.replace(".csv", "").split("_")
    dataset = partes[0].replace("hosp", "")
    imputer = partes[1]
    metrica_map = {"auc": "AUC", "sen": "Recall", "spe": "Specificity"}
    metrica = metrica_map.get(partes[2], partes[2])
    return dataset, imputer, metrica

def consolidar_planilhas(pasta_arquivos, arquivo_saida):
    arquivos = [f for f in os.listdir(pasta_arquivos) if f.endswith('.csv')]
    
    dados_consolidados = []
    
    for arquivo in arquivos:
        caminho_arquivo = os.path.join(pasta_arquivos, arquivo)
        df = pd.read_csv(caminho_arquivo)
        dataset, imputer, metrica = extrair_info_arquivo(arquivo)
        df['Arquivo'] = arquivo  # Adiciona uma coluna com o nome do arquivo
        df['Dataset'] = dataset
        df['Imputer'] = imputer
        df['Métrica'] = metrica
        dados_consolidados.append(df)
    
    df_final = pd.concat(dados_consolidados, ignore_index=True)
    df_final.to_csv(arquivo_saida, index=False)
    print(f"Arquivo consolidado salvo em: {arquivo_saida}")

# Exemplo de uso
pasta = "/home/filipe/Documentos/GitHub/Doutorado-real-datasets/results"
saida = "todos_resultados.csv"
consolidar_planilhas(pasta, saida)


Arquivo consolidado salvo em: todos_resultados.csv


# 2.0 **MACHINE LEARNING**

In [44]:
# gera arquivos consolidados no excel

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Pasta contendo os arquivos
results_folder = 'results/'
output_folder = 'output_excel/'
os.makedirs(output_folder, exist_ok=True)

# Lista de imputadores e algoritmos extraídos dos arquivos
imputadores = set()
algoritmos = ["SVM", "GB", "RF"]
hospitais = set()

for file in os.listdir(results_folder):
    if file.startswith("hosp") and "_auc.csv" in file:
        parts = file.split("_")
        hospitais.add(parts[0])  # Identifica os diferentes hospitais
        imputadores.add(parts[1])  # Considera apenas o tipo de imputador

# Dicionário para armazenar os resultados por hospital
df_hospitais = {}

# Criar um arquivo Excel para cada hospital
for hospital in sorted(hospitais):
    output_file = f'{output_folder}{hospital}_resultados.xlsx'
    with pd.ExcelWriter(output_file) as writer:
        df_hospitais[hospital] = {}
        
        for algoritmo in algoritmos:
            df_results = pd.DataFrame()
            
            for imputador in sorted(imputadores):
                hosp_prefix = f"{hospital}_{imputador}"  
                
                # Carregar os arquivos específicos para o imputador
                auc = pd.read_csv(f'{results_folder}{hosp_prefix}_auc.csv')
                sen = pd.read_csv(f'{results_folder}{hosp_prefix}_sen.csv')
                spe = pd.read_csv(f'{results_folder}{hosp_prefix}_spe.csv')
                
                # Criar DataFrame com média e desvio padrão lado a lado
                df_results[f'{imputador}_mean'] = [
                    auc[algoritmo].mean(),
                    sen[algoritmo].mean(),
                    spe[algoritmo].mean()
                ]
                df_results[f'{imputador}_std'] = [
                    auc[algoritmo].std(),
                    sen[algoritmo].std(),
                    spe[algoritmo].std()
                ]
            
            df_results.index = ["auc", "positive", "negative"]
            df_results.to_excel(writer, sheet_name=algoritmo)
            df_hospitais[hospital][algoritmo] = df_results

# Criar arquivo consolidado geral, incluindo os imputadores
output_file_geral = f'{output_folder}geral_resultados.xlsx'
with pd.ExcelWriter(output_file_geral) as writer:
    for algoritmo in algoritmos:
        df_geral_mean = pd.DataFrame()
        df_geral_std = pd.DataFrame()
        
        for imputador in sorted(imputadores):
            df_mean_temp = pd.DataFrame()
            df_std_temp = pd.DataFrame()
            
            for hospital in sorted(hospitais):
                df_hospital = df_hospitais[hospital][algoritmo]
                
                # Filtrar apenas colunas do imputador específico
                mean_col = f'{imputador}_mean'
                std_col = f'{imputador}_std'
                
                if mean_col in df_hospital.columns and std_col in df_hospital.columns:
                    df_mean_temp[hospital] = df_hospital[mean_col]
                    df_std_temp[hospital] = df_hospital[std_col]

            # Calcular a média dos hospitais para cada imputador
            df_geral_mean[imputador] = df_mean_temp.mean(axis=1)
            df_geral_std[imputador] = df_std_temp.mean(axis=1)

        # Concatenar os DataFrames de médias e desvios padrão
        df_geral = pd.concat([df_geral_mean, df_geral_std], axis=1, keys=['Mean', 'Std'])
        df_geral.to_excel(writer, sheet_name=algoritmo)

In [5]:
# gera boxplot OK

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Pasta contendo os arquivos
output_folder = 'output_excel/'
plots_folder = 'plots/'
os.makedirs(plots_folder, exist_ok=True)

# Lista de algoritmos e métricas
algoritmos = ["SVM", "GB", "RF"]
metricas = ["AUC", "Recall", "Specificity"]

# Identificar os arquivos
arquivos = [f for f in os.listdir(output_folder) if f.endswith("_resultados.xlsx")]

# Criar boxplots para cada algoritmo
for algoritmo in algoritmos:
    dados = []
    
    for arquivo in arquivos:
        hospital = arquivo.replace("_resultados.xlsx", "")
        df = pd.read_excel(f'{output_folder}{arquivo}', sheet_name=algoritmo, index_col=0)
        
        for imputador in df.columns:
            if "_mean" in imputador:
                metrica = imputador.replace("_mean", "")
                for i, valor in enumerate(df[imputador]):
                    dados.append({
                        "Hospital": hospital,
                        "Imputer": metrica,
                        "Metric": metricas[i],
                        "Valor": valor
                    })
    
    df_boxplot = pd.DataFrame(dados)
    
    # Criar e salvar os boxplots
    plt.figure(figsize=(12, 6))
    sns.boxplot(x="Metric", y="Valor", hue="Imputer", data=df_boxplot)
    plt.title(f'Imputer methods comparation - {algoritmo}')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f'{plots_folder}boxplot_{algoritmo}.png')
    plt.close()


In [23]:
import numpy as np
from Autoencoders.pmivae import ConfigVAE, PMIVAE

data = np.asarray([[31, 0.22, np.nan, 78], [0.43, np.nan, 67, 0.98]])

vae_config = ConfigVAE()

vae_config.verbose = 0
vae_config.epochs = 500
vae_config.neurons = [5]
vae_config.dropout_fc = [0.1]
vae_config.latent_dimension = 3
vae_config.input_shape = (4, )
vae_config.batch_size = 4

pmivae_model = PMIVAE(vae_config, num_samples=10)
print("[PMIVAE] Training and performing imputation...")
model = pmivae_model.fit(data)

data_imputed = model.transform(data)


[PMIVAE] Training and performing imputation...
1/1 [==============================] - 0s 36ms/step


In [24]:
print(data_imputed)

[[nan nan nan nan]
 [nan nan nan nan]]


# The end